# Notebook 00: manual_ec DIV30 Source Checkpoint

This notebook loads one supported source, records source availability, runs the requested `manual_ec` Scanpy QC/filter/preprocess checkpoint, and writes run-scoped tables and plots.

Supported forward sources:

- `cellranger_filtered`
- `cellbender_denoised`

CellBender files are treated as existing fixed inputs. This notebook does not run CellBender.


In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd


def env_bool(name: str, default: bool) -> bool:
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "y", "on"}


def env_tuple(name: str, default: tuple[str, ...]) -> tuple[str, ...]:
    raw = os.environ.get(name)
    if raw is None or not raw.strip():
        return default
    normalized = raw.replace(";", ",").replace(":", ",")
    return tuple(part.strip() for part in normalized.split(",") if part.strip())


# Make local package imports work whether the notebook is opened from repo root or notebooks/.
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    src_dir = candidate / "python_notebooks" / "src"
    if src_dir.exists():
        sys.path.insert(0, str(src_dir))
        break

from mge_organoid_python.data_sources import (
    Notebook00SourceConfig,
    find_repo_root,
    load_dataset_result,
    resolve_data_root,
)
from mge_organoid_python.notebook00_plots import (
    PlotConfig,
    plot_highest_expr_genes,
    plot_manual_ec_highly_variable_genes,
    plot_manual_ec_qc_scatter,
    plot_manual_ec_qc_violin,
    plot_manual_ec_source_comparison_cell_counts,
    plot_manual_ec_source_comparison_qc_metrics,
    plot_sample_counts,
    plot_source_availability,
)
from mge_organoid_python.notebook00_workflow import (
    ManualECFilterSettings,
    ManualECPreprocessSettings,
    apply_manual_ec_filter,
    calculate_qc_metrics,
    concat_samples,
    load_manual_ec_run_outputs,
    manual_ec_preprocess,
    manual_ec_qc_summary,
)


## Configuration

Use `NOTEBOOK00_ACTIVE_SOURCE=cellranger_filtered` for the primary run and `NOTEBOOK00_ACTIVE_SOURCE=cellbender_denoised` for the fixed existing-CellBender comparison run.


In [ ]:
REPO_ROOT = find_repo_root(Path.cwd())
DATA_ROOT = resolve_data_root()

ACTIVE_SOURCE = os.environ.get("NOTEBOOK00_ACTIVE_SOURCE", "cellranger_filtered")
SUPPORTED_FORWARD_SOURCES = {"cellranger_filtered", "cellbender_denoised"}
if ACTIVE_SOURCE not in SUPPORTED_FORWARD_SOURCES:
    raise ValueError(f"Notebook 00 manual_ec supports {sorted(SUPPORTED_FORWARD_SOURCES)}; got {ACTIVE_SOURCE!r}")

TARGET_DIVS = env_tuple("NOTEBOOK00_TARGET_DIVS", ("DIV30",))
TARGET_RUN_SAMPLE_IDS = env_tuple(
    "NOTEBOOK00_TARGET_RUN_SAMPLE_IDS",
    (
        "9853-MW-1",
        "9853-MW-2",
        "9853-MW-3",
        "9853-MW-4",
        "9853-MW-5",
        "9853-MW-6",
    ),
)
STRICT_MISSING_SOURCES = env_bool("NOTEBOOK00_STRICT_MISSING_SOURCES", False)
LOAD_MATRICES = env_bool("NOTEBOOK00_LOAD_MATRICES", True)
COMPARE_RUN_LABELS = env_tuple("NOTEBOOK00_COMPARE_RUN_LABELS", ())

RUN_LABEL = os.environ.get("NOTEBOOK00_RUN_LABEL") or f"{ACTIVE_SOURCE}_manual_ec_{'_'.join(TARGET_DIVS).lower()}_core_samples"
RUN_DIR = DATA_ROOT / "results" / "notebook00" / RUN_LABEL
TABLE_DIR = RUN_DIR / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

plot_config = PlotConfig.from_root(
    DATA_ROOT,
    run_label=RUN_LABEL,
    show=env_bool("NOTEBOOK00_SHOW_PLOTS", True),
    save=env_bool("NOTEBOOK00_SAVE_PLOTS", True),
)

print("REPO_ROOT:", REPO_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("ACTIVE_SOURCE:", ACTIVE_SOURCE)
print("TARGET_DIVS:", TARGET_DIVS)
print("TARGET_RUN_SAMPLE_IDS:", TARGET_RUN_SAMPLE_IDS)
print("STRICT_MISSING_SOURCES:", STRICT_MISSING_SOURCES)
print("LOAD_MATRICES:", LOAD_MATRICES)
print("COMPARE_RUN_LABELS:", COMPARE_RUN_LABELS)
print("RUN_DIR:", RUN_DIR)
print("PLOT_DIR:", plot_config.output_dir)
print("TABLE_DIR:", TABLE_DIR)


## Source Availability

This reports requested sample paths before loading matrices.


In [ ]:
source_config = Notebook00SourceConfig.from_defaults(
    data_source=ACTIVE_SOURCE,
    repo_root=REPO_ROOT,
    data_root=DATA_ROOT,
    target_divs=TARGET_DIVS,
    target_run_sample_ids=TARGET_RUN_SAMPLE_IDS,
    strict_missing_matrix_dirs=STRICT_MISSING_SOURCES,
)

source_preview = load_dataset_result(source_config, load_matrices=False)
source_table = source_preview.source_table
source_summary_df = source_preview.availability_summary()

source_table.to_csv(TABLE_DIR / "source_table.tsv", sep="	", index=False)
source_summary_df.to_csv(TABLE_DIR / "source_summary.tsv", sep="	", index=False)

display(source_table)
display(source_summary_df)

plot_source_availability(source_table, plot_config)
plot_sample_counts(source_table, plot_config)


## Load Available Samples

Only available samples are loaded when `STRICT_MISSING_SOURCES=False`. Set `NOTEBOOK00_LOAD_MATRICES=0` for source/report-only or comparison-only runs.


In [ ]:
if LOAD_MATRICES:
    dataset_result = load_dataset_result(source_config, load_matrices=True)
    adata_names = dataset_result.adata_names
    adata_list = dataset_result.adata_list
    source_table = dataset_result.source_table
else:
    dataset_result = source_preview
    adata_names = []
    adata_list = []
    source_table = source_preview.source_table.copy()
    print("LOAD_MATRICES is False; generated source reports only and skipped AnnData loading.")

source_table.to_csv(TABLE_DIR / "loaded_source_table.tsv", sep="	", index=False)

print("Loaded samples:", dataset_result.loaded_samples)
print("Skipped samples:", dataset_result.skipped_samples)
print("Number of AnnData objects:", len(adata_list))

loaded_shape_df = pd.DataFrame(
    {
        "run_sample_id": run_sample_id,
        "n_obs": one_sample_adata.n_obs,
        "n_vars": one_sample_adata.n_vars,
        "obs_names_unique": one_sample_adata.obs_names.is_unique,
        "data_source": one_sample_adata.obs["data_source"].iloc[0] if "data_source" in one_sample_adata.obs else ACTIVE_SOURCE,
    }
    for run_sample_id, one_sample_adata in zip(adata_names, adata_list)
)
loaded_shape_df.to_csv(TABLE_DIR / "loaded_sample_shapes.tsv", sep="	", index=False)
display(loaded_shape_df)


## manual_ec QC, Filtering, And Preprocessing

This is the requested Scanpy checkpoint: highest expressed genes, mitochondrial QC metrics, QC violins/scatters, `filter_cells(min_genes=20)`, `filter_genes(min_cells=3)`, manual biological/QC cutoffs, normalization, log1p, HVG selection, and HVG plot.


In [ ]:
manual_ec_adata = None
combined_adata = None

if adata_list:
    combined_adata = concat_samples(adata_names, adata_list)
    calculate_qc_metrics(combined_adata, mito_prefix="MT-", log1p=False, overwrite=True)
    manual_ec_qc_summary_loaded_df = manual_ec_qc_summary(combined_adata, stage="loaded")
    manual_ec_qc_summary_loaded_df.to_csv(TABLE_DIR / "manual_ec_qc_summary_loaded.tsv", sep="	", index=False)

    plot_highest_expr_genes(combined_adata, plot_config, n_top=20, name="highest_expr_genes_top20")
    plot_manual_ec_qc_violin(combined_adata, plot_config, name="manual_ec_qc_violin")
    plot_manual_ec_qc_scatter(
        combined_adata,
        plot_config,
        x="total_counts",
        y="pct_counts_mt",
        name="manual_ec_scatter_total_counts_pct_counts_mt",
    )
    plot_manual_ec_qc_scatter(
        combined_adata,
        plot_config,
        x="total_counts",
        y="n_genes_by_counts",
        name="manual_ec_scatter_total_counts_n_genes_by_counts",
    )

    filter_settings = ManualECFilterSettings()
    manual_ec_adata, manual_ec_filter_summary_df, manual_ec_filter_parameters_df = apply_manual_ec_filter(
        combined_adata,
        settings=filter_settings,
    )
    manual_ec_filter_summary_df.to_csv(TABLE_DIR / "manual_ec_filter_summary.tsv", sep="	", index=False)
    manual_ec_filter_parameters_df.to_csv(TABLE_DIR / "manual_ec_filter_parameters.tsv", sep="	", index=False)

    calculate_qc_metrics(manual_ec_adata, mito_prefix="MT-", log1p=False, overwrite=True)
    manual_ec_qc_summary_after_filter_df = manual_ec_qc_summary(manual_ec_adata, stage="manual_ec_filtered")
    manual_ec_qc_summary_after_filter_df.to_csv(
        TABLE_DIR / "manual_ec_qc_summary_after_filter.tsv",
        sep="	",
        index=False,
    )

    preprocess_settings = ManualECPreprocessSettings()
    manual_ec_preprocess_report_df, manual_ec_preprocess_parameters_df, manual_ec_hvg_genes_df = manual_ec_preprocess(
        manual_ec_adata,
        settings=preprocess_settings,
    )
    manual_ec_preprocess_report_df.to_csv(TABLE_DIR / "manual_ec_preprocess_report.tsv", sep="	", index=False)
    manual_ec_preprocess_parameters_df.to_csv(TABLE_DIR / "manual_ec_preprocess_parameters.tsv", sep="	", index=False)
    manual_ec_hvg_genes_df.to_csv(TABLE_DIR / "manual_ec_hvg_genes.tsv", sep="	", index=False)

    plot_manual_ec_highly_variable_genes(manual_ec_adata, plot_config, name="manual_ec_highly_variable_genes")

    # Keep the requested working object name available for interactive inspection.
    adata = manual_ec_adata

    display(manual_ec_filter_summary_df)
    display(manual_ec_qc_summary_after_filter_df)
    display(manual_ec_preprocess_report_df)
else:
    print("No AnnData objects loaded; skipping manual_ec matrix checkpoint.")
    pd.DataFrame().to_csv(TABLE_DIR / "manual_ec_filter_summary.tsv", sep="	", index=False)
    pd.DataFrame().to_csv(TABLE_DIR / "manual_ec_filter_parameters.tsv", sep="	", index=False)
    pd.DataFrame().to_csv(TABLE_DIR / "manual_ec_preprocess_report.tsv", sep="	", index=False)
    pd.DataFrame().to_csv(TABLE_DIR / "manual_ec_preprocess_parameters.tsv", sep="	", index=False)
    pd.DataFrame().to_csv(TABLE_DIR / "manual_ec_hvg_genes.tsv", sep="	", index=False)


## Optional manual_ec Source Comparison

Set `NOTEBOOK00_COMPARE_RUN_LABELS` to two completed manual_ec run labels, for example:

```text
cellranger_filtered_manual_ec_div30_core_samples:cellbender_denoised_manual_ec_div30_core_samples
```

Use `:` or `;` as the separator when passing labels through Slurm `--export`; comma-separated values also work inside the notebook. This reads run-scoped tables and writes comparison summaries/plots. It does not load expression matrices.


In [ ]:
if COMPARE_RUN_LABELS:
    comparison_outputs = load_manual_ec_run_outputs(DATA_ROOT / "results" / "notebook00", COMPARE_RUN_LABELS)
    for table_name, table_df in comparison_outputs.items():
        table_df.to_csv(TABLE_DIR / f"{table_name}.tsv", sep="	", index=False)
        display(table_df)

    plot_manual_ec_source_comparison_cell_counts(
        comparison_outputs["manual_ec_source_comparison_summary"],
        plot_config,
        name="manual_ec_source_comparison_cell_counts",
    )
    plot_manual_ec_source_comparison_qc_metrics(
        comparison_outputs["manual_ec_qc_metric_comparison_by_sample"],
        plot_config,
        metric="median_n_genes_by_counts",
        name="manual_ec_source_comparison_qc_metrics",
    )
else:
    print("NOTEBOOK00_COMPARE_RUN_LABELS is empty; skipping cross-run comparison.")


## Generated Objects

Main objects created by this notebook run:

- `source_table`: requested source paths and availability
- `adata_names`, `adata_list`: loaded per-sample objects
- `combined_adata`: concatenated loaded object before manual_ec filtering
- `manual_ec_adata` / `adata`: filtered, normalized, log-transformed, HVG-annotated object
- `TABLE_DIR`: saved TSV reports
- `plot_config.output_dir`: saved plot PNGs
